# 03 – Prédiction des cartes d'occupation du sol (2017, 2021, 2025)

On applique le modèle entraîné dans le notebook 02 sur les images complètes des trois années.

Points importants :
- même normalisation que pour l'entraînement (percentiles de l'image 2021, sauvegardés dans `normalisation.json`)
- le modèle travaille sur des patches de 128 px : on parcourt l'image avec une **fenêtre glissante avec recouvrement**, et on moyenne les probabilités en donnant plus de poids au centre des patches (les bords d'un patch sont moins bien prédits, faute de contexte)
- 2021 sert de contrôle : on peut comparer la prédiction à WorldCover sur toute la zone

Les cartes sont sauvegardées en GeoTIFF dans `resultats/cartes/` pour le notebook 04.

## 1. Configuration et chargement du modèle

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import rasterio
import tensorflow as tf
from tensorflow import keras

DATA_DIR = Path("data_saclay")
RES_DIR = Path("resultats")
CARTES_DIR = RES_DIR / "cartes"
CARTES_DIR.mkdir(parents=True, exist_ok=True)

ANNEES = ["2017", "2021", "2025"]
NOMS_CLASSES = ["Bâti", "Forêt / arbres", "Cultures", "Prairies / autre", "Eau"]
N_CLASSES = len(NOMS_CLASSES)
NODATA = 255
COULEURS = ListedColormap(["#d7191c", "#1a9641", "#fdae61", "#a6d96a", "#2b83ba"])

PATCH = 128
STRIDE = 64        # recouvrement de 50 % entre patches
BATCH = 64

model = keras.models.load_model(RES_DIR / "unet_saclay.keras")
with open(RES_DIR / "normalisation.json") as f:
    norm = json.load(f)
p_bas, p_haut = np.array(norm["p_bas"], "float32"), np.array(norm["p_haut"], "float32")
print("Modèle chargé :", model.name, f"({model.count_params():,} paramètres)")

## 2. Prédiction par fenêtre glissante

Pondération des patches : fenêtre de Hann 2D (≈ 1 au centre, proche de 0 sur les bords). L'image est complétée par réflexion (padding) pour que les bords de l'image soient aussi couverts par le centre d'un patch.

In [ ]:
def charger(annee):
    with rasterio.open(DATA_DIR / f"s2_{annee}.tif") as src:
        img = np.moveaxis(src.read().astype("float32"), 0, -1)
        profil = src.profile
    vide = (img == 0).all(axis=-1)
    img = np.clip((img - p_bas) / (p_haut - p_bas), 0, 1)
    img[vide] = 0
    return img, vide, profil


hann = np.hanning(PATCH + 2)[1:-1]
POIDS_PATCH = np.outer(hann, hann).astype("float32")[..., None]


def predire_image(img):
    H, W, _ = img.shape
    m = PATCH // 2
    # padding pour couvrir les bords + arrondi à un multiple du pas
    ph = m + (-(H + 2 * m - PATCH) % STRIDE)
    pw = m + (-(W + 2 * m - PATCH) % STRIDE)
    img_p = np.pad(img, ((m, ph), (m, pw), (0, 0)), mode="reflect")
    Hp, Wp, _ = img_p.shape

    somme = np.zeros((Hp, Wp, N_CLASSES), "float32")
    poids = np.zeros((Hp, Wp, 1), "float32")
    positions = [(y, x) for y in range(0, Hp - PATCH + 1, STRIDE) for x in range(0, Wp - PATCH + 1, STRIDE)]
    print(f"  {len(positions)} patches à prédire")

    for i in range(0, len(positions), BATCH):
        lot = positions[i:i + BATCH]
        X = np.stack([img_p[y:y + PATCH, x:x + PATCH] for y, x in lot])
        P = model.predict(X, verbose=0)
        for (y, x), p in zip(lot, P):
            somme[y:y + PATCH, x:x + PATCH] += p * POIDS_PATCH
            poids[y:y + PATCH, x:x + PATCH] += POIDS_PATCH

    proba = somme[m:m + H, m:m + W] / np.maximum(poids[m:m + H, m:m + W], 1e-6)
    return proba

In [ ]:
import time

cartes = {}
for annee in ANNEES:
    print(f"===== {annee} =====")
    t0 = time.time()
    img, vide, profil = charger(annee)
    proba = predire_image(img)
    carte = proba.argmax(-1).astype("uint8")
    carte[vide] = NODATA
    cartes[annee] = carte

    profil.update(count=1, dtype="uint8", nodata=NODATA, compress="deflate")
    with rasterio.open(CARTES_DIR / f"occupation_{annee}.tif", "w", **profil) as dst:
        dst.write(carte, 1)
    # confiance du modèle (proba max), utile pour repérer les zones douteuses
    profil_c = {**profil, "dtype": "float32", "nodata": None}
    with rasterio.open(CARTES_DIR / f"confiance_{annee}.tif", "w", **profil_c) as dst:
        dst.write(proba.max(-1), 1)
    print(f"  terminé en {time.time() - t0:.0f} s -> {CARTES_DIR / f'occupation_{annee}.tif'}")

## 3. Cartes obtenues

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, annee in zip(axes, ANNEES):
    ax.imshow(np.ma.masked_equal(cartes[annee], NODATA), cmap=COULEURS, vmin=0, vmax=4, interpolation="nearest")
    ax.set_title(f"Occupation du sol prédite — {annee}")
    ax.axis("off")
for i, nom in enumerate(NOMS_CLASSES):
    axes[-1].plot([], [], "s", color=COULEURS(i), label=nom, markersize=10)
axes[-1].legend(loc="lower right")
plt.tight_layout()
plt.savefig(RES_DIR / "cartes_occupation.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Répartition des classes par année

In [ ]:
surf_pixel_km2 = 10 * 10 / 1e6
print(f"{'Classe':18s}" + "".join(f"{a:>14s}" for a in ANNEES))
repartition = {}
for c, nom in enumerate(NOMS_CLASSES):
    ligne = []
    for annee in ANNEES:
        carte = cartes[annee]
        valides = carte != NODATA
        part = (carte[valides] == c).mean()
        ligne.append(part)
        repartition.setdefault(annee, {})[nom] = {"part": float(part), "km2": float((carte == c).sum() * surf_pixel_km2)}
    print(f"{nom:18s}" + "".join(f"{p:14.1%}" for p in ligne))

with open(RES_DIR / "repartition_classes.json", "w", encoding="utf-8") as f:
    json.dump(repartition, f, indent=2, ensure_ascii=False)

## 5. Contrôle sur 2021 : prédiction vs WorldCover

Sur 2021, on peut comparer la prédiction à WorldCover sur toute la zone (attention : la majorité des pixels ont servi à l'entraînement, ce score est donc optimiste ; le vrai score de généralisation est celui du jeu de test dans le notebook 02).

In [ ]:
with rasterio.open(DATA_DIR / "labels_2021.tif") as src:
    wc = src.read(1)

valide = (wc != NODATA) & (cartes["2021"] != NODATA)
accord = (wc[valide] == cartes["2021"][valide]).mean()
print(f"Accord global prédiction 2021 / WorldCover : {accord:.1%}")
for c, nom in enumerate(NOMS_CLASSES):
    inter = ((wc == c) & (cartes["2021"] == c) & valide).sum()
    union = (((wc == c) | (cartes["2021"] == c)) & valide).sum()
    print(f"  IoU {nom:18s} {inter / max(union, 1):.3f}")

## 6. Zoom sur le plateau de Saclay

Comparaison visuelle image / carte prédite sur le secteur du campus (zone de forte construction entre 2017 et 2025).

In [ ]:
# Fenêtre autour du campus Paris-Saclay (à ajuster si besoin) : coordonnées en pixels
from rasterio.warp import transform

with rasterio.open(DATA_DIR / "s2_2021.tif") as src:
    ligne, col = src.index(*transform("EPSG:4326", src.crs, [2.17], [48.712]))
    ligne, col = ligne[0], col[0]
T = 400  # 4 km
zone = (slice(max(ligne - T // 2, 0), ligne + T // 2), slice(max(col - T // 2, 0), col + T // 2))

def rgb(annee):
    img, _, _ = charger(annee)
    return np.clip(img[zone][..., [2, 1, 0]] * 1.2, 0, 1)

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for j, annee in enumerate(["2017", "2025"]):
    axes[0, j].imshow(rgb(annee))
    axes[0, j].set_title(f"Sentinel-2 {annee}")
    axes[1, j].imshow(np.ma.masked_equal(cartes[annee][zone], NODATA), cmap=COULEURS, vmin=0, vmax=4, interpolation="nearest")
    axes[1, j].set_title(f"Prédiction {annee}")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.savefig(RES_DIR / "zoom_saclay.png", dpi=120, bbox_inches="tight")
plt.show()

## Observations

*(à compléter après exécution : cohérence visuelle des cartes, évolution des surfaces par classe, zones où le modèle semble se tromper)*